### Exploratory Notebook

In [2]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType
from pyspark.sql.functions import from_json, col
from pyspark.sql.functions import regexp_replace


################################################
# Schemas 

# Define the schema for the JSON data
event_schema = StructType([
    StructField("age_of_insured", IntegerType(), True),
    StructField("coverage_amount", DoubleType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("policy_id", StringType(), True),
    StructField("policy_type", StringType(), True),
    StructField("premium_amount", DoubleType(), True),
    StructField("region", StringType(), True),
])

# Define the schema for the policy_type field
policy_type_schema = StructType([
                StructField("type", StringType(), True),
                StructField("brand", StringType(), True),
])

########################################################
# Load and clean data
     
# Read the JSON data from the specified path and apply the schema
invalid_json_df =   spark \
                    .read\
                    .format("json")\
                    .schema(event_schema) \
                    .load("/Volumes/ageas/bronze/files")

# Fix the invalid JSON in the policy_type field by replacing single quotes with double quotes
valid_json_df = invalid_json_df.withColumn(
    "policy_type",
    regexp_replace(  
        regexp_replace(col("policy_type"), "'", "\""),
        "None", 
        "\"null\""
    )
)

# Replace policy_type string with a valid JSON object
valid_json_df = valid_json_df.withColumn(
    "policy_type_object",
    from_json(col("policy_type"), policy_type_schema)
)


valid_json_df = valid_json_df.selectExpr(
    "age_of_insured",
    "coverage_amount",
    "customer_id",
    "event_timestamp",
    "event_type",
    "policy_id",
    "policy_type_object.type AS policy_type",
    "policy_type_object.brand AS policy_brand",
    "premium_amount",
    "region"
)#.filter(col("policy_id") == "POL-60927")

display(valid_json_df)

ModuleNotFoundError: No module named 'pyspark'

In [42]:
silver_df = spark.table("ageas.ageas.silver_policy_event")
display(silver_df)

,age_of_insured,coverage_amount,customer_id,event_timestamp,event_type,policy_id,policy_type,policy_brand,premium_amount,region,ingestion_date_time,source_system,age_of_insured_band
0,56,61020.73,CUS-70018,2024-01-21T10:40:41.087Z,claim,POL-58051,auto,LifeSecure,938.88,North,2026-08-12 10:57:06.972,policy_events_api,50-59
1,62,31329.27,CUS-56010,2024-07-24T01:56:56.088Z,purchase,POL-84577,life,InsureCorp,687.09,East,2026-08-12 10:57:06.972,policy_events_api,60-69
2,39,26810.22,CUS-4315,2024-12-22T10:25:58.088Z,purchase,POL-58297,life,InsureCorp,1108.62,West,2026-08-12 10:57:06.972,policy_events_api,30-39
3,66,83055.63,CUS-37602,2024-04-23T23:51:29.088Z,claim,POL-21521,auto,LifeSecure,1766.21,South,2026-08-12 10:57:06.972,policy_events_api,60-69
4,71,38446.22,CUS-84286,2024-09-30T03:07:48.088Z,purchase,POL-69237,auto,InsureCorp,905.40,North,2026-08-12 10:57:06.972,policy_events_api,70-79
5,55,29914.22,CUS-39540,2024-03-25T20:25:51.088Z,purchase,POL-60927,None,LifeSecure,581.55,North,2026-08-12 10:57:06.972,policy_events_api,50-59
6,58,43995.57,CUS-89696,2024-08-13T06:52:57.088Z,purchase,POL-47666,health,LifeSecure,1272.79,South,2026-08-12 10:57:06.972,policy_events_api,50-59
7,60,44422.96,CUS-8242,2024-03-16T11:19:41.088Z,claim,POL-16907,None,LifeSecure,793.39,North,2026-08-12 10:57:06.972,policy_events_api,60-69
8,75,14040.10,CUS-52665,2024-01-08T18:23:04.088Z,cancellation,POL-54380,home,InsureCorp,1714.52,West,2026-08-12 10:57:06.972,policy_events_api,70-79
9,71,43324.32,CUS-94911,2024-03-27T09:38:29.088Z,cancellation,POL-45657,health,ProtectPlus,525.92,West,2026-08-12 10:57:06.972,policy_events_api,70-79
